In [ ]:
pip install rouge-score

In [ ]:
pip install -U sentence-transformers

In [1]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/shubhammishra/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
from tqdm import tqdm
import csv
import time

In [3]:
from rouge_score import rouge_scorer
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.meteor_score import meteor_score as ms

In [4]:
import pandas as pd
import numpy as np

In [5]:
def calculate_score(text,reference):
    # define the texts to compare
    # tokenize the texts and reference
    text_tokens = word_tokenize(text)
    reference_tokens = word_tokenize(reference)

    # initialize the Rouge scorer
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    # calculate the Rouge 1, Rouge 2, and Rouge L scores
    rouge_scores = scorer.score(text, reference)

    # print the Rouge scores
    rouge_1=rouge_scores['rouge1'].fmeasure
    rouge_2=rouge_scores['rouge2'].fmeasure
    rouge_l=rouge_scores['rougeL'].fmeasure
    
    # calculate the BLEU score
    bleu_score = sentence_bleu([reference_tokens], text_tokens)

    # print the BLEU score
#     print("BLEU:", bleu_score)

    # tokenize the texts for Meteor
    text_tokens = [token.lower() for token in text_tokens]
    reference_tokens = [token.lower() for token in reference_tokens]
    # calculate the Meteor score
    meteor_score = ms([text_tokens], reference_tokens)

    # print the Meteor score
#     print("Meteor:", meteor_score)
    return rouge_1,rouge_2,rouge_l,bleu_score,meteor_score

In [22]:
data=pd.read_csv('')

In [7]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

In [8]:
def get_mpnet_embedding(text:str)->list[float]:
    return (model.encode(text))
def get_mpnet_similarity(text1,text2):
    embedding1 = (get_mpnet_embedding(text1))
    embedding2 = (get_mpnet_embedding(text2))
    cosine_sim = np.dot(embedding1, embedding2) / (np.linalg.norm(embedding1) * np.linalg.norm(embedding2))
    return cosine_sim

In [24]:
rouge_1=[]
rouge_2=[]
rouge_l=[]
bleu_score=[]
meteor_score=[]
cos_sim=[]
for i in tqdm(range(len(data))):
    text=data['answer_generated'][i]
    reference=data['ground_truth'][i]
    tupl=calculate_score(text,reference)
    rouge_1.append(tupl[0])
    rouge_2.append(tupl[1])
    rouge_l.append(tupl[2])
    bleu_score.append(tupl[3])
    meteor_score.append(tupl[4])
    cos_sim.append(get_mpnet_similarity(text,reference))
    time.sleep(2)

 62%|██████▏   | 31/50 [01:10<00:42,  2.26s/it]/opt/homebrew/lib/python3.10/site-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
100%|██████████| 50/50 [01:53<00:00,  2.27s/it]


In [25]:
data['rouge_1']=rouge_1
data['rouge_2']=rouge_2
data['rouge_l']=rouge_l
data['bleu_score']=bleu_score
data['mpnet_score']=cos_sim

In [26]:
data.head(2)

,Unnamed: 0,title,question,ground_truth,answer_generated,rouge_1,rouge_2,rouge_l,meteor_score,mpnet_score,score,bleu_score
0,0,Submission of original documents to police for...,I have filed FIR via Magistrate. While submitt...,Dear Client When the police has asked you to b...,"As a legal advisor, I must inform you that the...",0.120930,0.018692,0.093023,0.040323,0.534247,3,2.385053e-155
1,1,How to handle fake police threatening ?,How to handle fake police threatening ??,Dear Client In case you are receiving fake pol...,The question asked is how to handle fake polic...,0.147982,0.022523,0.089686,0.053435,0.765976,2,2.579656e-155


In [27]:
new_cols=['title','question','ground_truth','answer_generated','rouge_1','rouge_2','rouge_l','bleu_score','mpnet_score','score']

In [28]:
data=data[new_cols]

In [29]:
data.to_csv('')